**Clonning the repo to use the dataset prepared**

In [8]:
!git clone https://github.com/amanmandal7/COMP8410-A1_Evaluation_Presentation.git
%cd COMP8410-A1_Evaluation_Presentation/pair2_diffusion

Cloning into 'COMP8410-A1_Evaluation_Presentation'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 45 (delta 6), reused 37 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 3.67 MiB | 9.87 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/COMP8410-A1_Evaluation_Presentation/pair2_diffusion/COMP8410-A1_Evaluation_Presentation/pair2_diffusion


In [9]:
# Installing transformers
!pip install transformers -q

In [10]:
from transformers import pipeline
import os
import time

**Laoding both the models and doing a test run**

In [11]:
# Loading both the models
model_a = pipeline("image-classification", model="Organika/sdxl-detector")
model_b = pipeline("image-classification", model="dima806/deepfake_vs_real_image_detection")
print("both models loaded successfully!")

Loading weights:   0%|          | 0/425 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

both models loaded successfully!


In [12]:
# run once on one image to see what labels each model returns
real_files = sorted(os.listdir("data/real"))
sample_img = os.path.join("data/real", real_files[0])

print("Model A output:", model_a(sample_img))
print("Model B output:", model_b(sample_img))

Model A output: [{'label': 'artificial', 'score': 0.9998877048492432}, {'label': 'human', 'score': 0.0001122595786000602}]
Model B output: [{'label': 'Real', 'score': 0.9875906705856323}, {'label': 'Fake', 'score': 0.012409337796270847}]


**Evaluating both the Models**

In [13]:
def evaluate(model, folder, real_label, fake_label):
    preds = {}
    files = sorted(os.listdir(folder))

    for fname in files:
        full_path = os.path.join(folder, fname)
        try:
            out = model(full_path)
            top_label = out[0]["label"]      # highest-score label

            # normalise things to "real" or "fake"
            if top_label == real_label:
                preds[fname] = "real"
            elif top_label == fake_label:
                preds[fname] = "fake"
            else:
                preds[fname] = top_label
        except Exception as err:
            print(f"Error processing {fname}: {err}")
            preds[fname] = "error"

    return preds

# Model A: human = real, artificial = fake
print("\n=== Model A (AI-generated image detector) ===")
a_real = evaluate(model_a, "data/real", "human", "artificial")
a_fake = evaluate(model_a, "data/fake", "human", "artificial")

print("on REAL images:", a_real)
print("on FAKE images:", a_fake)

# Model B: Real = real, Fake = fake
print("\n=== Model B (deepfake detector) ===")
b_real = evaluate(model_b, "data/real", "Real", "Fake")
b_fake = evaluate(model_b, "data/fake", "Real", "Fake")

print("on REAL images:", b_real)
print("on FAKE images:", b_fake)


=== Model A (AI-generated image detector) ===
on REAL images: {'real_1.jpeg': 'fake', 'real_2.jpeg': 'fake', 'real_3.jpeg': 'fake', 'real_4.jpeg': 'fake', 'real_5.jpeg': 'fake'}
on FAKE images: {'fake_1.jpeg': 'fake', 'fake_2.jpeg': 'fake', 'fake_3.jpeg': 'fake', 'fake_4.jpeg': 'real', 'fake_5.jpeg': 'fake'}

=== Model B (deepfake detector) ===
on REAL images: {'real_1.jpeg': 'real', 'real_2.jpeg': 'fake', 'real_3.jpeg': 'real', 'real_4.jpeg': 'real', 'real_5.jpeg': 'fake'}
on FAKE images: {'fake_1.jpeg': 'fake', 'fake_2.jpeg': 'real', 'fake_3.jpeg': 'fake', 'fake_4.jpeg': 'real', 'fake_5.jpeg': 'real'}


**Computing the metrics for both the Models**

In [14]:
def metrics(real_preds, fake_preds):
    correct_real = 0
    for v in real_preds.values():
        if v == "real":
            correct_real += 1

    correct_fake = 0
    for v in fake_preds.values():
        if v == "fake":
            correct_fake += 1

    total = len(real_preds) + len(fake_preds)
    accuracy = (correct_real + correct_fake) / total

    # fake called real (false accept)
    false_accepts = 0
    for v in fake_preds.values():
        if v == "real":
            false_accepts += 1

    # real called fake (false reject)
    false_rejects = 0
    for v in real_preds.values():
        if v == "fake":
            false_rejects += 1

    return accuracy, false_accepts, false_rejects

acc_a, fa_a, fr_a = metrics(a_real, a_fake)
acc_b, fa_b, fr_b = metrics(b_real, b_fake)

print("\n--- Final Results ---")
print(f"Model A -> accuracy: {acc_a:.2f}, false accepts: {fa_a}, false rejects: {fr_a}")
print(f"Model B -> accuracy: {acc_b:.2f}, false accepts: {fa_b}, false rejects: {fr_b}")


--- Final Results ---
Model A -> accuracy: 0.40, false accepts: 1, false rejects: 5
Model B -> accuracy: 0.50, false accepts: 3, false rejects: 2
